# Reviewed anchor repair trial

Nine isolated comparisons. Production geometry and hydraulic inputs are unchanged.

This trial tests six missing graph cuts and three reviewed node correspondences (two identity repairs, one already correct). It does **not** regenerate widths, slopes, K, X or hydrographs. Independent states remain canonical; island deduplication is separate.

Run all cells. Change `EXAMPLE` and `STATE` below to inspect another case. No ipywidgets is required.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import geopandas as gpd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RAPID/src/rapid_tools').exists())
sys.path.insert(0, str(ROOT / 'RAPID/src'))
sys.path.insert(0, str(ROOT / 'RAPID/investigations'))
from rapid_tools import collapse_creation_review as review
from rapid_tools.anchor_repair_trial import node_mapping
from test_reviewed_anchor_repairs import plot
OUT = ROOT / 'RAPID/outputs/anchor_repair_trial_20261008'
summary = pd.DataFrame(json.loads((OUT / 'summary.json').read_text()))
display(summary[['example','state','inserted_cuts','identity_changes','missing_identity_before','missing_identity_after']])

In [ ]:
EXAMPLE = '41'
STATE = 'S016_unit_4'
row = review.case_row(EXAMPLE, STATE)
folder = OUT / f'{EXAMPLE}_{STATE}'
frame = pd.read_parquet(row.frame_path)
mapping = node_mapping(frame)
before = review.vectors(row.child_links_path,'links')[['id_link','geometry']].merge(frame[['id_link','id_us_node','id_ds_node','u','v']], on='id_link', validate='one_to_one')
nodes = review.vectors(row.child_nodes_path,'nodes').copy()
nodes['parent_identity'] = nodes.id_node.map(mapping)
after = gpd.read_parquet(folder / 'trial_links.parquet')
new_nodes = gpd.read_parquet(folder / 'trial_nodes.parquet')
fig = plot(before, after, review.vectors(row.base_links_path,'links'), nodes, new_nodes,
           review.vectors(row.base_nodes_path,'nodes'), review.int_ids(row.required_boundary_nodes),
           review.int_ids(row.selected_parent_link_ids), f'{EXAMPLE} — {STATE}: reviewed repair trial')
fig.show()

In [ ]:
display(pd.read_json(folder / 'actions.json'))
case_summary = json.loads((folder / 'summary.json').read_text())
print('Unreviewed boundary correspondences:')
display(pd.DataFrame(case_summary['remaining_unreviewed_correspondences']))
exterior = pd.read_csv(folder / 'exterior_comparison.csv')
display(exterior[(~exterior.endpoint_pair_present_before) | (~exterior.endpoint_pair_present_after)])

## Interpretation

- In 41 unit 4, the two exterior fragments recover the parent geometry exactly; the middle fragment is 665.14 m long.
- A recovered identity at a displaced node is **not** a coordinate snap. Geometry offsets remain explicit in the exterior table.
- An absent direct exterior endpoint pair can represent a subdivided path, not necessarily disconnection. The trial preserves every original directed link as a chain and leaves all network connections intact.
- 02 unit 16 retains unreviewed P18/P20, 15 unit 144 retains P291, and the 02 unit 5 control retains unreviewed P9/P12.
- Before routing, fragment widths and hydraulic parameters must be regenerated and the policy for preserving exterior properties checked. Whole-link parameters must not simply be copied onto all fragments.